## 6. Training the Model (Putting it all together)
Now that we have built the Forward Pass, we are ready to implement **Backpropagation** and **Gradient Descent** to actually train this model!

First, we need to create our **Training Dataset**. A dataset is just pairs of `(Input, Target)` words. 
For example, if the input is `"Order"`, the correct target is `"Shipment"`.

# Building a Neural Network from Scratch (Forward Pass)
In this assignment, we will build a Neural Network entirely from scratch using only `numpy`! We are not going to use any high-level libraries like PyTorch or TensorFlow, because we want to understand the **math** behind the magic.

We are building a **Next Word Predictor** based on a small vocabulary of states from a purchase-order lifecycle:
`Order -> Shipment -> Receive -> Restock -> Inventory -> Forecast -> Order`

Our goal is to create a network that, if you give it the word `Order`, it should mathematically predict `Shipment`.

## 1. The Vocabulary
Computers don't understand English words like 'Order' or 'Shipment'. They only understand numbers. 
So, the first thing we must do is create a dictionary that maps each word to a unique number (an **Index**).

In [ ]:
import numpy as np

# Our tiny vocabulary
states = [
    "Order",
    "Shipment",
    "Receive",
    "Restock",
    "Inventory",
    "Forecast",
    "Scenario",
    "Cabinet",
    "Drug",
    "Invoice"
]

# Map each word to an ID (e.g., 'Order' -> 0)
state_to_id = {state: i for i, state in enumerate(states)}
# Map each ID back to a word (e.g., 0 -> 'Order')
id_to_state = {i: state for state, i in state_to_id.items()}

vocabulary_size = len(states)
print(f"Our vocabulary has {vocabulary_size} words.")
print("ID of 'Order':", state_to_id["Order"])

Our vocabulary has 10 words.
ID of 'Order': 0


## 2. One-Hot Encoding
Even though 'Order' is now the number `0`, we can't just feed the number `0` into our network's math equations.

**Why can not we just send a single number?**
Let's pretend our network just multiplies the input by a weight of `0.5`. If we feed in our IDs directly:
- `Order` (0) -> `0 * 0.5 = 0`
- `Shipment` (1) -> `1 * 0.5 = 0.5`
- `Receive` (2) -> `2 * 0.5 = 1.0`

The network's math forces a relationship that doesn't exist. It thinks `Receive` is mathematically 'bigger' or 'stronger' than `Shipment`. But these are just words! A 'Receive' is not mathematically twice as big as a 'Shipment'. The numbers are just arbitrary ID badges.

**The Solution (Separate Channels)**
Instead of using a single number that goes up and down, we use **One-Hot Encoding** to give every single word its own independent channel (like a separate switch). We convert the ID into an array of zeroes, where only one specific position is "hot" (set to `1`).

In [ ]:
def one_hot(index, vocab_size):
    """Converts an index into a one-hot vector."""
    vec = np.zeros(vocab_size)
    vec[index] = 1.0
    return vec

example_word = "Shipment"
idx = state_to_id[example_word]
one_hot_vector = one_hot(idx, vocabulary_size)

print(f"Word: {example_word}")
print(f"Index: {idx}")
print(f"One-Hot Vector: {one_hot_vector}")

Word: Shipment
Index: 1
One-Hot Vector: [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]


## 3. The Neural Network Brain (Weights & Biases)
Our network has two layers of connections:
1. **Input -> Hidden Layer**: This acts as an "Embedding" layer. It takes our giant one-hot vector and squeezes it down into a smaller, dense "hidden" vector. 
2. **Hidden Layer -> Output**: This takes the hidden vector and expands it back out to give a score for every single word in our vocabulary.

Because the network hasn't "learned" anything yet, we initialize these weights with **random numbers**.

In [ ]:
np.random.seed(42) # So we get the same random numbers every time

hidden_size = 8

# LAYER 1: Input -> Hidden
# W1 shape: (vocab_size, hidden_size)
W1 = np.random.randn(vocabulary_size, hidden_size) * 0.1
b1 = np.zeros(hidden_size)

# LAYER 2: Hidden -> Output
# W2 shape: (hidden_size, vocab_size)
W2 = np.random.randn(hidden_size, vocabulary_size) * 0.1
b2 = np.zeros(vocabulary_size)

print("Weights are initialized randomly!")

Weights are initialized randomly!


## 4. The Softmax Function
At the very end of our network, we get a list of "raw scores" (called **Logits**). These scores can be anything: `+3.4`, `-1.2`, `0.0`.
To make sense of these scores, we convert them into **Probabilities** (percentages that add up to 100%). We do this using the **Softmax** function.

In [ ]:
def softmax(logits):
    """Converts raw scores into probabilities that sum to 1."""
    # We subtract the max value for numerical stability (to avoid math errors with huge exponentials)
    shifted_logits = logits - np.max(logits)
    exponentials = np.exp(shifted_logits)
    return exponentials / np.sum(exponentials)

## 5. The Forward Pass (Putting it all together)
Now we do the exact sequence of math operations to predict a word.

1. **Input**: Convert the word to a one-hot vector.
2. **Hidden Pre-Activation**: `(Input * W1) + b1`
3. **Activation**: We apply `np.tanh()`. This adds "curves" to our math (non-linearity), and keeps the values balanced between `-1` and `1`.
4. **Logits**: `(Hidden * W2) + b2`
5. **Probabilities**: Apply `softmax()` to the logits.

In [ ]:
def predict_next_word(input_word):
    # 1. Input encoding
    idx = state_to_id[input_word]
    x = one_hot(idx, vocabulary_size)
    
    # 2 & 3. Hidden layer with Tanh activation
    hidden = np.tanh(np.dot(x, W1) + b1)
    
    # 4. Output layer (Logits)
    logits = np.dot(hidden, W2) + b2
    
    # 5. Probabilities
    probs = softmax(logits)
    
    # Find the word with the highest probability
    predicted_idx = np.argmax(probs)
    confidence = probs[predicted_idx]
    predicted_word = id_to_state[predicted_idx]
    
    return predicted_word, confidence

# Let's test it out!
word = "Order"
guess, conf = predict_next_word(word)
print(f"If the input is '{word}', the totally untrained model guesses: '{guess}' (Confidence: {conf:.1%})")
print("Since we haven't trained the weights yet, the guess is basically completely random!")

If the input is 'Order', the totally untrained model guesses: 'Restock' (Confidence: 10.3%)
Since we haven't trained the weights yet, the guess is basically completely random!


## 6. Training the Model (Putting it all together)
Now that we have built the Forward Pass, we are ready to implement **Backpropagation** and **Gradient Descent** to actually train this model!

First, we need to create our **Training Dataset**. A dataset is just pairs of `(Input, Target)` words. 
For example, if the input is `"Order"`, the correct target is `"Shipment"`.

In [ ]:
# Our training sequence: Order -> Shipment -> Receive -> Restock -> Inventory -> Forecast -> Order
# We will create pairs of (Input, Target)
training_data = [
    ("Order", "Shipment"),
    ("Shipment", "Receive"),
    ("Receive", "Restock"),
    ("Restock", "Inventory"),
    ("Inventory", "Forecast"),
    ("Forecast", "Order")
]

# We will reserve the rest for testing later
test_data = [
    ("Cabinet", "Drug"),
    ("Drug", "Invoice")
]

print(f"Created {len(training_data)} training pairs and {len(test_data)} test pairs.")

Created 6 training pairs and 2 test pairs.


### The Complete Neural Network (Training & Backpropagation)
We will now combine everything we learned into a clean, professional code block. We will create helper functions for each layer so you can see exactly how a neural network is structured in production!

*Note: For the heavy math derivations behind `backward_pass()` and the Loss function, please refer to **Section 17-19** in your `PREREQUISITE_KNOWLEDGE.md` guide!*

In [ ]:
# --- HYPERPARAMETERS ---
epochs = 500
learning_rate = 0.1
hidden_size = 8

# --- HELPER FUNCTIONS ---
def init_network(vocab_size, hidden_size):
    np.random.seed(42)
    W1 = np.random.randn(vocab_size, hidden_size) * 0.1
    b1 = np.zeros(hidden_size)
    W2 = np.random.randn(hidden_size, vocab_size) * 0.1
    b2 = np.zeros(vocab_size)
    return W1, b1, W2, b2

def forward_pass(x, W1, b1, W2, b2):
    # Hidden Layer (Tanh)
    hidden = np.tanh(np.dot(x, W1) + b1)
    # Output Layer (Logits -> Softmax)
    logits = np.dot(hidden, W2) + b2
    probs = softmax(logits)
    return hidden, probs

def compute_loss(probs, true_label_idx):
    # Cross-Entropy Loss
    # We only care about the probability the network gave to the CORRECT answer.
    # If correct answer is 1.0 (100%), loss is 0. If it's 0.01 (1%), loss is huge!
    correct_prob = probs[true_label_idx]
    # Small number added to avoid log(0)
    loss = -np.log(correct_prob + 1e-9)
    return loss

def backward_pass(x, true_idx, probs, hidden, W2):
    # 1. Output Layer Error (Prediction - Reality)
    d_logits = probs.copy()
    d_logits[true_idx] -= 1.0 # The magic of Cross-Entropy + Softmax derivative
    
    # Gradients for W2 and b2
    dW2 = np.outer(hidden, d_logits)
    db2 = d_logits
    
    # 2. Hidden Layer Error (Sending error backwards through W2 and Tanh)
    d_hidden = np.dot(W2, d_logits) * (1 - hidden**2)
    
    # Gradients for W1 and b1
    dW1 = np.outer(x, d_hidden)
    db1 = d_hidden
    
    return dW1, db1, dW2, db2

def update_weights(W1, b1, W2, b2, dW1, db1, dW2, db2, lr):
    # Gradient Descent: Take a step DOWNHILL (subtracting the gradient)
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2
    return W1, b1, W2, b2

print("Neural Network Architecture loaded and ready for training!")

Neural Network Architecture loaded and ready for training!


In [ ]:
# 1. Initialize fresh weights
W1, b1, W2, b2 = init_network(vocabulary_size, hidden_size)

print("Starting Training Loop...\n")

# 2. The Training Loop
for epoch in range(epochs):
    total_loss = 0
    
    # We train on one pair at a time (Stochastic Gradient Descent with batch_size=1)
    for input_word, target_word in training_data:
        # A. Setup Input and Target
        x = one_hot(state_to_id[input_word], vocabulary_size)
        true_idx = state_to_id[target_word]
        
        # B. Forward Pass (Make a guess)
        hidden, probs = forward_pass(x, W1, b1, W2, b2)
        
        # C. Compute Error (Loss)
        loss = compute_loss(probs, true_idx)
        total_loss += loss
        
        # D. Backward Pass (Who is to blame?)
        dW1, db1, dW2, db2 = backward_pass(x, true_idx, probs, hidden, W2)
        
        # E. Update Weights (Gradient Descent)
        W1, b1, W2, b2 = update_weights(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate)
        
    # Print progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        avg_loss = total_loss / len(training_data)
        print(f"Epoch {epoch + 1}/{epochs} | Average Loss: {avg_loss:.4f}")

print("\nTraining Complete! The Loss has reached the bottom of the bowl!")

In [ ]:
# 3. Testing the Model (Did it learn?)
print("--- TRAINING DATA RESULTS ---")
for input_word, target_word in training_data:
    x = one_hot(state_to_id[input_word], vocabulary_size)
    hidden, probs = forward_pass(x, W1, b1, W2, b2)
    
    pred_idx = np.argmax(probs)
    pred_word = id_to_state[pred_idx]
    confidence = probs[pred_idx]
    
    print(f"Input: '{input_word}' | Predicted: '{pred_word}' (Confidence: {confidence:.1%}) | Expected: '{target_word}'")
    
print("\n--- TEST DATA RESULTS (Unseen data!) ---")
for input_word, target_word in test_data:
    x = one_hot(state_to_id[input_word], vocabulary_size)
    hidden, probs = forward_pass(x, W1, b1, W2, b2)
    
    pred_idx = np.argmax(probs)
    pred_word = id_to_state[pred_idx]
    confidence = probs[pred_idx]
    
    print(f"Input: '{input_word}' | Predicted: '{pred_word}' (Confidence: {confidence:.1%}) | Expected: '{target_word}'")
    if pred_word != target_word:
        print(f"   -> It failed here because it has NEVER seen '{input_word}' during training!")

--- TRAINING DATA RESULTS ---
Input: 'Order' | Predicted: 'Restock' (Confidence: 10.3%) | Expected: 'Shipment'
Input: 'Shipment' | Predicted: 'Receive' (Confidence: 10.4%) | Expected: 'Receive'
Input: 'Receive' | Predicted: 'Order' (Confidence: 10.6%) | Expected: 'Restock'
Input: 'Restock' | Predicted: 'Restock' (Confidence: 10.3%) | Expected: 'Inventory'
Input: 'Inventory' | Predicted: 'Receive' (Confidence: 10.4%) | Expected: 'Forecast'
Input: 'Forecast' | Predicted: 'Scenario' (Confidence: 10.5%) | Expected: 'Order'

--- TEST DATA RESULTS (Unseen data!) ---
Input: 'Cabinet' | Predicted: 'Restock' (Confidence: 10.3%) | Expected: 'Drug'
   -> It failed here because it has NEVER seen 'Cabinet' during training!
Input: 'Drug' | Predicted: 'Scenario' (Confidence: 10.3%) | Expected: 'Invoice'
   -> It failed here because it has NEVER seen 'Drug' during training!
